---
format:
  html:
    embed-resources: true
    code-overflow: wrap
execute:
  enabled: false
---

# Lab 4: Data is Delicious
Shiqi Wu

## Part 1 — Option A: Web Scraping

I chose Low Carb Meal Plan (11) from Tastes Better from Scratch because I am interested in its variety of low-carb dinner recipes.

Source: https://tastesbetterfromscratch.com/low-carb-meal-plan-11/

I will use BeautifulSoup to collect the day of the week, recipe name, recipe link, and estimated recipe cost, and organize the results into a pandas DataFrame. Each row will represent one dinner in the Monday–Friday meal plan.

The prices are the website's estimated costs per recipe, not costs per serving or current ingredient prices.

In [12]:
import requests
import pandas as pd
from bs4 import BeautifulSoup

In [13]:
filename = "Low Carb Meal Plan (11) _ - Tastes Better From Scratch.html"

with open(filename, encoding="utf-8") as file:
    html = file.read()

soup = BeautifulSoup(html, "html.parser")
soup.title.get_text()

'Low Carb Meal Plan (11) | - Tastes Better From Scratch'

In [14]:
days = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday"]
meal_paragraphs = []

for paragraph in soup.find_all("p"):
    text = paragraph.get_text(" ", strip=True)
    if text.startswith(tuple(days)):
        meal_paragraphs.append(paragraph)

len(meal_paragraphs)

5

In [25]:
meals = []

for paragraph in meal_paragraphs:
    text = paragraph.get_text(" ", strip=True)
    day = text.split(":")[0].strip()

    recipe = paragraph.find("a")
    name = recipe.get_text(strip=True)
    link = recipe["href"]

    price = float(text.split("$")[-1].strip())

    meals.append({
        "Day of the Week": day,
        "Name of Recipe": name,
        "Link to Recipe": link,
        "Price of Recipe": price
    })

df_meals = pd.DataFrame(meals)

with pd.option_context("display.max_colwidth", None):
    display(df_meals)

,Day of the Week,Name of Recipe,Link to Recipe,Price of Recipe
0,Monday,Cashew Chicken Wraps,https://tastesbetterfromscratch.com/chicken-cashew-crunch-wraps/,8.83
1,Tuesday,Mixed Green Salad with Berries,https://tastesbetterfromscratch.com/mixed-green-salad-with-berries/,18.03
2,Wednesday,Cedar Plank Salmon,https://tastesbetterfromscratch.com/cedar-plank-salmon/,14.84
3,Thursday,Steak Kabobs,https://tastesbetterfromscratch.com/marinated-steak-kebabs/,19.08
4,Friday,Taco Soup,https://tastesbetterfromscratch.com/taco-soup/,12.08


### Data Collection Note

Direct requests to the meal-plan page returned HTTP 403 with a message requiring JavaScript and cookies. I therefore saved the page using my browser and parsed the saved HTML with BeautifulSoup.

The weekday, recipe name, recipe link, and estimated recipe cost were extracted from the HTML rather than entered manually. The saved source HTML must be kept in the same folder as this notebook to rerun Part 1.

## Part 2 — Option A: Data from an API

Monday's recipe in my selected meal plan is Cashew Chicken Wraps. I will search the Tasty API using the keyword "cashew chicken" to find similar recipes without limiting the results to wraps.

I will retrieve up to the first 100 matching results and organize the returned data into a pandas DataFrame.

In [16]:
monday_recipe = df_meals.loc[df_meals["Day of the Week"] == "Monday", "Name of Recipe"].iloc[0]
search_query = monday_recipe.lower().replace(" wraps", "")

print("Monday's recipe:", monday_recipe)
print("Search keyword:", search_query)

Monday's recipe: Cashew Chicken Wraps
Search keyword: cashew chicken


In [19]:
from getpass import getpass

api_key = getpass("Enter your RapidAPI key: ")

In [20]:
recipes_url = "https://tasty.p.rapidapi.com/recipes/list"

headers = {
    "X-RapidAPI-Key": api_key,
    "X-RapidAPI-Host": "tasty.p.rapidapi.com"
}

params = {"from": 0, "size": 40, "q": search_query}

response = requests.get(recipes_url, headers=headers, params=params, timeout=30)
response.status_code

200

In [21]:
recipe_data = response.json()
recipes = recipe_data["results"]

print("Total matching results:", recipe_data["count"])
print("Results received:", len(recipes))

Total matching results: 5
Results received: 5


In [22]:
df_recipes = pd.json_normalize(recipes)

print("Rows and columns:", df_recipes.shape)
df_recipes.head()

Rows and columns: (5, 72)


,nutrition_visibility,country,instructions,keywords,facebook_posts,language,seo_path,id,brand,slug,...,nutrition.updated_at,nutrition.protein,nutrition.fat,nutrition.calories,nutrition.sugar,show.name,show.id,total_time_tier.tier,total_time_tier.display_tier,total_time_tier
0,auto,ZZ,"[{'start_time': 1160, 'appliance': None, 'end_...",NaN,[],und,"8757513,9295873,64448",2412,None,cashew-chicken-stir-fry,...,2024-09-03T03:10:44+02:00,37.0,16.0,423.0,14.0,Goodful,34,under_30_minutes,Under 30 minutes,NaN
1,auto,US,"[{'start_time': 58000, 'appliance': None, 'end...",,[],eng,"8757513,9295873,64452",7718,None,tasty-101-butter-chicken-chicken-makhani,...,2024-05-01T08:02:38+02:00,44.0,36.0,649.0,15.0,Tasty 101,63,NaN,NaN,NaN
2,auto,US,"[{'start_time': 0, 'appliance': None, 'end_tim...","20-minute salad, asian chicken salad, chinese ...",[],eng,"8757513,9295873,64448",9522,None,garlicky-chinese-chicken-salad,...,2025-10-25T08:00:41+02:00,25.0,30.0,513.0,14.0,Tasty,17,under_30_minutes,Under 30 minutes,NaN
3,auto,US,"[{'start_time': 0, 'appliance': None, 'end_tim...",,[],eng,"8757513,9295873,64452",7393,None,chicken-pulao,...,2024-09-03T03:10:44+02:00,34.0,15.0,676.0,1.0,Tasty,17,under_1.5_hours,Under 1.5 hours,NaN
4,auto,US,"[{'start_time': 4000, 'appliance': None, 'end_...",,[],eng,"9295813,64486,64459,9299650",8210,None,malaysian-chicken-curry-laksa-laksa-lemak,...,NaN,NaN,NaN,NaN,NaN,Tasty,17,under_1_hour,Under 1 hour,NaN


In [23]:
df_recipes[["id", "name", "slug"]]

,id,name,slug
0,2412,Cashew Chicken Stir-Fry,cashew-chicken-stir-fry
1,7718,How To Make The Creamiest Butter Chicken,tasty-101-butter-chicken-chicken-makhani
2,9522,Garlicky Chinese Chicken Salad,garlicky-chinese-chicken-salad
3,7393,Chicken Pulao,chicken-pulao
4,8210,Malaysian Chicken Curry Laksa (Laksa Lemak),malaysian-chicken-curry-laksa-laksa-lemak


### Data Quality Check

In [26]:
print("Meal plan rows:", len(df_meals))
print("Missing meal plan values:", df_meals.isna().sum().sum())
print("API recipes received:", len(df_recipes))
print("Unique API recipe IDs:", df_recipes["id"].nunique())

Meal plan rows: 5
Missing meal plan values: 0
API recipes received: 5
Unique API recipe IDs: 5


### Results

The Tasty API reported 5 matches for the search term "cashew chicken", and all 5 were returned in the first request. No additional pages were needed. After normalization, the DataFrame contained 5 rows and 72 columns.

Based on the recipe names, Cashew Chicken Stir-Fry is the most directly related result to Monday's Cashew Chicken Wraps. Both names refer to cashew chicken, although the dishes have different preparations. The other returned recipes are broader chicken-related results rather than exact matches. I retained all five API results in the DataFrame.